# Governed Gold inspection
Copyright © 2026 Innovation In Software Corporation. All rights reserved.

Import this optional notebook into Snowflake as ADV_ANALYST_US using ADV_WH. Its role guard prevents an accidentally elevated session from reading Gold. Run each cell after the instructor. The notebook expands the distinction between table freshness and consumer authorization; it is not required for the SQL consumer lab.
Import into ADV_ORDERS.CONSUMER as ADV_ANALYST_US, choose ADV_WH as the query warehouse, and use the Snowflake warehouse Python runtime. Confirm the active role before executing the queries.


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
role = session.sql("SELECT CURRENT_ROLE()").collect()[0][0]
assert role == "ADV_ANALYST_US", "Select the US analyst role before reading Gold"
role

## Contract and authority
The secure view exposes current order state. Row access limits the region, and masking limits the contact column. The notebook execution identity determines these policies; its visual location in an app does not establish access rights.

In [ ]:
session.sql("SELECT ORDER_ID, STATUS, AMOUNT, REGION, EMAIL FROM ADV_ORDERS.CONSUMER.ORDERS ORDER BY ORDER_ID").to_pandas()

## Business meaning
Active order value excludes cancelled orders and sums the full amount of each latest order. It is neither the sum of all events nor recognized revenue. Expected here: two active US orders with total USD 165. Authorization and correctness are separate checks.

In [ ]:
session.sql("SELECT COUNT(*) ACTIVE_ORDERS, SUM(AMOUNT) ACTIVE_AMOUNT_USD FROM ADV_ORDERS.CONSUMER.ORDERS WHERE STATUS <> 'CANCELLED'").to_pandas()

## Freshness and recovery
A successful SELECT confirms a consumer read. It does not prove that the upstream Dynamic Table has refreshed since the latest source change. The engineer inspects refresh history and reconciles Gold. If data looks stale, record the observation and ask the instructor to show that evidence; do not grant the analyst direct Silver access.